# Heterogeneous GATv2 over a QLoRA FLAN-T5 encoder

Joint polarity and intensity classification of opinion expressions on MPQA 2.0, evaluated with 5-fold cross-validation.

## Installation

In [ ]:
%pip install -q torch_geometric "transformers>=4.51" peft bitsandbytes accelerate spacy scikit-learn scipy sentencepiece tqdm
!python -m spacy download en_core_web_sm

## Data

The corpus is not distributed with this code. Put the two preprocessed JSON files in `data/` (or set `DATA_DIR`):

- `mpqa2_samples.json`: object with a `csds_objects` list
- `mpqa2_folds.json`: `IDs_{trainset,validationset,testset}_fold_{1..5}` id lists

## Config

In [ ]:
import os, re, gc, json, copy, glob, time, random, statistics, contextlib
import numpy as np
import torch

SEED = 7
def set_seed(seed=None):
    seed = SEED if seed is None else seed
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

DATA_DIR = os.environ.get('DATA_DIR', 'data')
OUTPUT_DIR = os.environ.get('OUTPUT_DIR', 'outputs')
FILE_PATH = os.path.join(DATA_DIR, 'mpqa2_samples.json')
FOLD_FILE_PATH = os.path.join(DATA_DIR, 'mpqa2_folds.json')
CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, 'checkpoints')
RESULTS_DIR = os.path.join(OUTPUT_DIR, 'results')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Which model family
# 'hsgat'           : encoder + heterogeneous GATv2 + readout + heads
# 'encoder_softmax' : baseline, QLoRA encoder + mean pooling + softmax heads
# 'text2text'       : baseline, QLoRA FLAN-T5 generating "polarity: .. ; intensity: .."
MODEL_TYPE = 'hsgat'
EXPERIMENT = 'full'

# Encoder catalog (FLAN-T5 only)
QUANT_META = ('absmax', 'quant_map', 'nested_absmax', 'nested_quant_map', 'quant_state')
MODEL_CATALOG = {
    # lr: peak learning rate of the LoRA/norm parameters; effective batch size is 16 for every encoder.
    'large': dict(id='google/flan-t5-large', d_model=1024, lr=2e-4, bs=8, ga=2),
    'xl':    dict(id='google/flan-t5-xl',    d_model=2048, lr=2e-4, bs=4, ga=4),
    'xxl':   dict(id='google/flan-t5-xxl',   d_model=4096, lr=1e-4, bs=2, ga=8),   # ~12-14 GB VRAM in 4-bit
}
MODEL_SIZE = 'xl'                   # 'large', 'xl' or 'xxl'

# 4-bit QLoRA
LORA_RANK, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.1
LORA_TARGETS = ["q", "v", "k", "o", "wi_0", "wi_1", "wo"]

# Architecture
HIDDEN_DIM = 256
DROPOUT_RATE = 0.15
MAX_SEQ_LEN = 512
GNN_LAYERS, GNN_HEADS = 2, 2           # GNN_LAYERS = 0 removes the GNN but keeps the readout

# Graph structure (ablation knobs)
# Relation families: 'syntax' (token-token dependency), 'span' (token -> span),
# 'source' (NESTED_IN, EXPRESSES, MENTIONED_BY), 'target' (belongs_to_target, targets, directed_at).
EDGE_FAMILIES = {'syntax', 'span', 'source', 'target'}
USE_REVERSE_EDGES = True
# Nested-source handling. Sources are listed from outermost to innermost.
#   'nested'    : one node per source, NESTED_IN chain, innermost source expresses the span (default)
#   'flat'      : one node per source, no NESTED_IN chain edges
#   'innermost' : only the innermost source node
#   'none'      : a single anonymous source node, no mention edges
SOURCE_MODE = 'nested'
SOURCE_DEPTH_EMB = False               # learned embedding of the depth (0 = innermost) added to source nodes
MAX_SRC_DEPTH = 4

# Polarity
POS_IDX, NEG_IDX, NEUTRAL_IDX = 0, 1, 2
NUM_POL_CLASSES = 3
NEUTRAL_BOOST = 1.25
USE_HIER_POL = True
HIER_GATE_WEIGHT, HIER_DIR_WEIGHT = 0.4, 0.4       # lambda_g, lambda_d
POL_COMBINE_ALPHA = 0.3                            # alpha
CALIBRATE_NEUTRAL = True
CALIB_GRID = np.round(np.arange(-0.5, 0.5 + 1e-9, 0.05), 2)   # 21 candidate biases
SEL_W_POL_WEIGHTED, SEL_W_POL_MACRO = 0.5, 0.5

# Intensity
LOW_IDX, MEDIUM_IDX, HIGH_IDX = 0, 1, 2
NUM_INT_CLASSES = 3
USE_ORDINAL_INT = True
ORD_T1_WEIGHT, ORD_T2_WEIGHT = 0.4, 0.4            # lambda_t1, lambda_t2 (equal to gate/direction weights)
INT_COMBINE_ALPHA = 0.4                            # alpha_u
USE_EMD_INT = True
EMD_WEIGHT = 0.15                                  # lambda_emd
SEL_W_INT_STD, SEL_W_INT_PAPER = 0.5, 0.5

# Training
EPOCHS = 8
EARLY_STOP_PATIENCE = 3                           # stop when neither task improved on validation for this many epochs (0 = off)
NUM_FOLDS = 5
FOLDS_TO_RUN = list(range(NUM_FOLDS))              # e.g. [4] to run only the fifth fold
GNN_LR = 5e-4                                      # GNN layers, readouts and task heads
WEIGHT_DECAY_ENC, WEIGHT_DECAY_HEAD = 1e-2, 0.01
WARMUP_RATIO, COSINE_ETA_MIN_RATIO = 0.05, 0.1
GRAD_CLIP = 1.0

# Numerics and convergence
# bf16 where the GPU supports it; otherwise fp16 with dynamic loss scaling
# (small gradients underflow in fp16 without it).
_BF16_OK = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
COMPUTE_DTYPE = torch.bfloat16 if _BF16_OK else torch.float16
USE_GRAD_SCALER = bool(torch.cuda.is_available() and not _BF16_OK)
# T5 does not normalize the residual stream between blocks, so the last four layers differ in scale;
# layer-norm each one (no parameters) before mixing.
NORMALIZE_LAYER_MIX = True

# Faster and stronger learning (all switchable, see EXPERIMENTS)
# TASK_SPECIFIC_BRANCHES: shared encoder and GNN, but a separate readout and fusion block per task.
# USE_SPAN_SKIP: direct encoder-to-head path; the mean of the aspect tokens (prompt prefix) and of the
#   expression tokens inside the sentence is concatenated with the sentence mean.
# USE_EMA: moving average of the trainable weights, used for validation, checkpointing and testing.
TASK_SPECIFIC_BRANCHES = True
USE_SPAN_SKIP = True
USE_EMA = True
EMA_HALF_LIFE_EPOCHS = 0.5      # half-life of the average, in epochs (decay per step is derived from it)

# Intensity: label noise at the boundaries and decision calibration
# INT_SOFT_BOUNDARY: 'low-medium' / 'medium-high' get a half/half target over the two neighbouring
#   classes (flat head, ordinal thresholds and EMD term), matching the boundary-tolerant metric.
# CALIBRATE_INTENSITY: validation-tuned log-offsets for 'medium' and 'high' (relative to 'low'),
#   reported next to the standard result.
INT_SOFT_BOUNDARY = True
CALIBRATE_INTENSITY = True
INT_CALIB_GRID = np.round(np.arange(-0.6, 0.6 + 1e-9, 0.1), 2)

# Gold opinion type as an input (not a prediction target)
# The gold type (agreement, arguing, subjectivity, intention, sentiment) is an input to both tasks.
#   USE_GOLD_TYPE : append "| type= <type>" to the prompt
#   USE_TYPE_EMB  : learned type embedding added to each branch before its head (HS-GAT only)
# There is no type head and no type loss.
USE_GOLD_TYPE = True
USE_TYPE_EMB = True
TYPE_EMB_DIM = 32
NUM_TYPE_CLASSES = 5
TYPE_IDS = {'agreement': 0, 'arguing': 1, 'subjectivity': 2, 'intention': 3, 'sentiment': 4}
TYPE_WORDS = {'agreement': 'agreement', 'arguing': 'arguing', 'expressive_subjectivity': 'subjectivity',
              'expressive subjectivity': 'subjectivity', 'intention': 'intention', 'sentiment': 'sentiment'}

# Primary task weights; auxiliary budgets are matched: lambda_g + lambda_d == lambda_t1 + lambda_t2.
TASK_WEIGHT_POL, TASK_WEIGHT_INT = 1.0, 1.0
LABEL_SMOOTHING_POL = 0.05
LABEL_SMOOTHING_INT = 0.05

# Text-to-text baseline
T2T_MAX_NEW_TOKENS = 16
T2T_INSTRUCTION = "Classify the polarity (positive, negative, neutral) and the intensity (low, medium, high) of the opinion expression."
POL_WORDS = ['positive', 'negative', 'neutral']
INT_WORDS = ['low', 'medium', 'high']

# Labels
POLARITY_MAP = {'positive': 0, 'negative': 1, 'neutral': 2}
# 3-way intensity: training target and standard-metric label space.
INTENSITY_MAP = {'low': 0, 'low-medium': 0, 'medium': 1,
                 'high': 2, 'medium-high': 2, 'extreme': 2, 'high-extreme': 2}
# 5-way intensity: used ONLY by the boundary-tolerant evaluation metric.
INTENSITY_5CAT_MAP = {'low': 0, 'low-medium': 1, 'medium': 2,
                      'medium-high': 3, 'high': 4, 'extreme': 4, 'high-extreme': 4}
# Each 5-way category as a (low, medium, high) multi-hot vector (Ahmadnia et al., 2024).
INTENSITY_5CAT_MULTIHOT = {0: (1, 0, 0), 1: (1, 1, 0), 2: (0, 1, 0),
                           3: (0, 1, 1), 4: (0, 0, 1)}

# Experiment registry
# Each entry overrides the defaults above. Select with EXPERIMENT_QUEUE = ['name'] or the
# environment variable HSGAT_EXPERIMENTS="name1,name2".
EXPERIMENTS = {
    'full': {},
    # edge-family ablations
    'edges_syntax_only':      dict(EDGE_FAMILIES={'syntax'}),
    'edges_no_source':        dict(EDGE_FAMILIES={'syntax', 'span', 'target'}),
    'edges_no_target':        dict(EDGE_FAMILIES={'syntax', 'span', 'source'}),
    'edges_no_source_target': dict(EDGE_FAMILIES={'syntax', 'span'}),
    'edges_no_reverse':       dict(USE_REVERSE_EDGES=False),
    # nested-source ablations
    'src_flat':       dict(SOURCE_MODE='flat'),
    'src_innermost':  dict(SOURCE_MODE='innermost'),
    'src_none':       dict(SOURCE_MODE='none'),
    'src_depth_emb':  dict(SOURCE_DEPTH_EMB=True),
    # structure / baselines
    'no_gnn':             dict(GNN_LAYERS=0),
    'baseline_softmax':   dict(MODEL_TYPE='encoder_softmax'),
    'baseline_text2text': dict(MODEL_TYPE='text2text'),
    # intensity-head ablations
    'int_flat_only':    dict(USE_ORDINAL_INT=False, USE_EMD_INT=False),
    'int_flat_emd':     dict(USE_ORDINAL_INT=False),
    'int_flat_ordinal': dict(USE_EMD_INT=False),
    # speed/strength components
    'shared_branches': dict(TASK_SPECIFIC_BRANCHES=False),
    'no_span_skip':    dict(USE_SPAN_SKIP=False),
    'no_ema':          dict(USE_EMA=False),
    'int_hard_labels': dict(INT_SOFT_BOUNDARY=False),
    'no_gold_type':    dict(USE_GOLD_TYPE=False),
    'type_prompt_only': dict(USE_TYPE_EMB=False),
    # polarity and task-balance ablations
    'no_hier_pol':      dict(USE_HIER_POL=False),
    'unbalanced_int03': dict(TASK_WEIGHT_INT=0.3),
    # diagnostic: does intensity training limit polarity? (no intensity gradient at all)
    'polarity_only':    dict(TASK_WEIGHT_INT=0.0, ORD_T1_WEIGHT=0.0, ORD_T2_WEIGHT=0.0, EMD_WEIGHT=0.0),
}
EXPERIMENT_QUEUE = ['full']
SEED_LIST = [42]

# Snapshot of the defaults, so experiments can be applied one after another in one session.
_NO_RESET = {'EXPERIMENTS', 'EXPERIMENT_QUEUE', 'SEED_LIST'}
_DEFAULT_CFG = {k: copy.deepcopy(v) for k, v in list(globals().items())
                if k.isupper() and not k.startswith('_') and k not in _NO_RESET}


def _derive():
    """Quantities that depend on MODEL_SIZE, MODEL_TYPE and EXPERIMENT."""
    g = globals()
    g['ENC'] = MODEL_CATALOG[MODEL_SIZE]
    g['BATCH_SIZE'], g['GRAD_ACCUM'] = g['ENC']['bs'], g['ENC']['ga']
    g['ENC_BASE_LR'] = g['ENC']['lr']
    prefix = {'hsgat': 'HS-GAT', 'encoder_softmax': 'EncSoftmax', 'text2text': 'Text2Text'}[MODEL_TYPE]
    g['MODEL_NAME'] = f"{prefix}-flant5-{MODEL_SIZE}-{EXPERIMENT}"


def apply_experiment(name):
    """Reset every config value to its default, then apply the named overrides."""
    assert name in EXPERIMENTS, f"unknown experiment '{name}'. Choices: {list(EXPERIMENTS)}"
    globals().update(copy.deepcopy(_DEFAULT_CFG))
    globals().update(copy.deepcopy(EXPERIMENTS[name]))
    globals()['EXPERIMENT'] = name
    _derive()
    if abs((HIER_GATE_WEIGHT + HIER_DIR_WEIGHT) - (ORD_T1_WEIGHT + ORD_T2_WEIGHT)) > 1e-9 \
            or TASK_WEIGHT_POL != TASK_WEIGHT_INT:
        print(f"[CONFIG] Note: task weights are not balanced in experiment '{name}'.")
    print(f"[CONFIG] experiment={name} | {MODEL_NAME} | d_model={ENC['d_model']} | "
          f"BS={BATCH_SIZE}x{GRAD_ACCUM}={BATCH_SIZE*GRAD_ACCUM}")


os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
_derive()
set_seed()

## Data Loader

In [ ]:
import spacy
from tqdm import tqdm
from torch_geometric.data import HeteroData, Dataset
from transformers import AutoTokenizer

nlp, tokenizer = None, None


def init_tools():
    """Load spaCy and the tokenizer for the current config (called after apply_experiment)."""
    global nlp, tokenizer
    if nlp is None and MODEL_TYPE != 'text2text':
        print("[DATA] Loading spaCy...")
        nlp = spacy.load("en_core_web_sm")
    if tokenizer is None or tokenizer.name_or_path != ENC['id']:
        tokenizer = AutoTokenizer.from_pretrained(ENC['id'], use_fast=True)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token


def _clean_heads(raw):
    out = []
    if isinstance(raw, list):
        for o in raw:
            ch = o.get('clean_head', '') if isinstance(o, dict) else o
            if str(ch).strip():
                out.append(str(ch).strip())
    return out


def load_all_data():
    with open(FILE_PATH, 'r', encoding='utf-8') as f:
        raw_data = json.load(f)
    samples = raw_data.get('csds_objects', [])
    id_to_sample, skipped = {}, 0
    for item in tqdm(samples, desc="Processing", ncols=100):
        sample_id = item.get('unique_id')
        if not sample_id or not item.get('text') or not item.get('head'):
            skipped += 1; continue
        pol_raw, int_raw = item.get('polarity'), item.get('intensity')
        if not pol_raw or not int_raw:
            skipped += 1; continue
        pol = POLARITY_MAP.get(str(pol_raw).lower().strip())
        int_key = str(int_raw).lower().strip()
        intensity, intensity5 = INTENSITY_MAP.get(int_key), INTENSITY_5CAT_MAP.get(int_key)
        if pol is None or intensity is None or intensity5 is None:
            skipped += 1; continue
        id_to_sample[sample_id] = {
            'id': sample_id, 'text': str(item['text']).strip(),
            'span_text': str(item['head']).strip(),
            'span_start_char': item.get('head_start', -1),
            'span_end_char': item.get('head_end', -1),
            'polarity': pol, 'intensity': intensity, 'intensity5': intensity5,
            'sources': _clean_heads(item.get('nested_source', [])) or ['Author'],
            'targets': _clean_heads(item.get('target', [])),
            'type_word': TYPE_WORDS.get(str(item.get('annotation_type') or '').lower().strip(), 'sentiment')}
    print(f"[DATA] Skipped: {skipped}, Valid: {len(id_to_sample)}")
    return id_to_sample


def load_fold_splits():
    with open(FOLD_FILE_PATH, 'r', encoding='utf-8') as f:
        fd = json.load(f)
    return [{'train_ids': fd.get(f'IDs_trainset_fold_{i}', []),
             'val_ids': fd.get(f'IDs_validationset_fold_{i}', []),
             'test_ids': fd.get(f'IDs_testset_fold_{i}', [])} for i in range(1, 6)]


def find_span_token_indices(doc, sample, max_words):
    s, e = sample['span_start_char'], sample['span_end_char']
    if s != -1 and e != -1:
        for token in doc:
            if token.idx == s:
                valid = [i for i in range(token.i, min(len(doc), token.i + 10))
                         if doc[i].idx + len(doc[i].text) <= e]
                if valid: return valid
    sw = sample['span_text'].split()
    if not sw: return []
    for i in range(len(doc)):
        if doc[i].text.lower() == sw[0].lower():
            if all(i + j < len(doc) and doc[i + j].text.lower() == w.lower() for j, w in enumerate(sw)):
                return [i + k for k in range(len(sw)) if i + k < max_words]
    return []


def find_target_token_indices(doc, span_word_indices, targets, max_words):
    if targets:
        for tgt in targets:
            tw = tgt.lower().split()
            if not tw: continue
            for i in range(len(doc)):
                if doc[i].text.lower() == tw[0]:
                    if all(i + j < len(doc) and doc[i + j].text.lower() == w.lower() for j, w in enumerate(tw)):
                        return [i + k for k in range(len(tw)) if i + k < max_words]
    # Heuristic proxy target: objects (and their modifiers) of the verb governing the span.
    if not span_word_indices: return []
    mv = next((i for i in span_word_indices if i < len(doc) and doc[i].pos_ in ['VERB', 'AUX']), None)
    if mv is None:
        mv = next((doc[i].head.i for i in span_word_indices if i < len(doc)
                   and doc[doc[i].head.i].pos_ in ['VERB', 'AUX']), None)
    if mv is None: return []
    out = []
    obj_d = {'obj', 'dobj', 'iobj', 'pobj', 'attr', 'oprd', 'dative'}
    mod_d = {'amod', 'det', 'compound', 'nmod', 'poss', 'nummod'}
    for t in doc:
        if t.head.i == mv and t.dep_ in obj_d:
            out.append(t.i); out += [c.i for c in t.children if c.dep_ in mod_d]
    return sorted(set(i for i in out if i < max_words))


# Relation -> family. A relation is built only if its family is enabled; reverse relations
# (prefix 'rev_') additionally require USE_REVERSE_EDGES.
REL_FAMILY = {
    'syntax': 'syntax',
    'belongs_to': 'span', 'rev_belongs': 'span',
    'NESTED_IN': 'source', 'EXPRESSES': 'source', 'MENTIONED_BY': 'source',
    'rev_NESTED_IN': 'source', 'rev_EXPRESSES': 'source', 'rev_MENTIONED_BY': 'source',
    'belongs_to_target': 'target', 'targets': 'target', 'directed_at': 'target',
    'rev_belongs_to_target': 'target', 'rev_targets': 'target', 'rev_directed_at': 'target',
}
# 8 forward relations + 7 reverse relations. SYNTAX is already added in both
# directions, so it has no separate reverse relation.
ALL_EDGE_TYPES = [
    ('token', 'syntax', 'token'), ('token', 'belongs_to', 'span'), ('span', 'rev_belongs', 'token'),
    ('span_source', 'NESTED_IN', 'span_source'), ('span_source', 'EXPRESSES', 'span'),
    ('token', 'MENTIONED_BY', 'span_source'), ('token', 'belongs_to_target', 'target'),
    ('span', 'targets', 'target'), ('span_source', 'directed_at', 'target'),
    ('span_source', 'rev_NESTED_IN', 'span_source'), ('span', 'rev_EXPRESSES', 'span_source'),
    ('span_source', 'rev_MENTIONED_BY', 'token'), ('target', 'rev_belongs_to_target', 'token'),
    ('target', 'rev_targets', 'span'), ('target', 'rev_directed_at', 'span_source'),
]


def rel_active(rel):
    if REL_FAMILY[rel] not in EDGE_FAMILIES: return False
    if rel.startswith('rev_') and not USE_REVERSE_EDGES: return False
    return True


def active_edge_types():
    return [et for et in ALL_EDGE_TYPES if rel_active(et[1])]


def select_sources(sources):
    """Apply SOURCE_MODE to the outermost-to-innermost source list."""
    if SOURCE_MODE == 'innermost': return sources[-1:]
    if SOURCE_MODE == 'none': return ['Author']
    return sources


def build_hetero_graph(sample):
    """Prompt: aspect=<expression> | sentence=<sentence> [| type=<gold type>, only if USE_GOLD_TYPE]"""
    doc = nlp(sample['text'])
    tokens = [t.text for t in doc] or ["[UNK]"]
    expr_words = sample['span_text'].split() or ["[UNK]"]

    prefix = ["aspect="] + expr_words + ["|", "sentence="]
    combined = prefix + tokens
    if USE_GOLD_TYPE:
        combined = combined + ["|", "type=", sample['type_word']]

    enc = tokenizer(combined, is_split_into_words=True, truncation=True, max_length=MAX_SEQ_LEN)
    full_ids, wids = enc['input_ids'], enc.word_ids()

    ss = len(prefix)
    w2s = {}
    for i, w in enumerate(wids):
        if w is not None and ss <= w < ss + len(doc):
            w2s.setdefault(w - ss, []).append(i)
    first_sub = {w: s[0] for w, s in w2s.items()}

    # Edges
    syntax_edges = []
    for t in doc:
        if t.head.i != t.i and t.i in first_sub and t.head.i in first_sub:
            syntax_edges += [[first_sub[t.i], first_sub[t.head.i]],
                             [first_sub[t.head.i], first_sub[t.i]]]
    span_w = find_span_token_indices(doc, sample, len(doc))
    belongs = [[sub, 0] for w in span_w if w in w2s for sub in w2s[w]]
    sources = select_sources(sample['sources']); nsrc = len(sources)
    nested = [[i, i - 1] for i in range(1, nsrc)] if SOURCE_MODE == 'nested' else []
    expresses = [[nsrc - 1, 0]] if nsrc else []
    mention = [] if SOURCE_MODE == 'none' else \
        [[first_sub[t.i], si] for si, nm in enumerate(sources)
         for t in doc if t.text.lower() == nm.lower().split()[0] and t.i in first_sub]
    tgt_w = find_target_token_indices(doc, span_w, sample['targets'], len(doc))
    has_tgt = len(tgt_w) > 0
    tgt_belongs = [[sub, 0] for w in tgt_w if w in w2s for sub in w2s[w]]
    targets_e = [[0, 0]] if has_tgt else []
    directed = [[nsrc - 1, 0]] if (has_tgt and nsrc) else []

    data = HeteroData()
    data['token'].input_ids = torch.tensor(full_ids, dtype=torch.long)
    data['token'].attention_mask = torch.ones(len(full_ids), dtype=torch.long)
    data['token'].num_nodes = len(full_ids)
    data['span'].num_nodes = 1;           data['span'].dummy = torch.zeros(1, 1)
    data['span_source'].num_nodes = nsrc; data['span_source'].dummy = torch.zeros(nsrc, 1)
    data['target'].num_nodes = 1;         data['target'].dummy = torch.zeros(1, 1)
    data['target'].has_real_target = torch.tensor([has_tgt], dtype=torch.bool)

    def set_e(s, r, d, e):
        if not rel_active(r): return                      # ablated relation family
        data[s, r, d].edge_index = (torch.tensor(e, dtype=torch.long).t().contiguous()
                                    if e else torch.empty((2, 0), dtype=torch.long))
    set_e('token', 'syntax', 'token', syntax_edges)
    set_e('token', 'belongs_to', 'span', belongs)
    set_e('span_source', 'NESTED_IN', 'span_source', nested)
    set_e('span_source', 'EXPRESSES', 'span', expresses)
    set_e('token', 'MENTIONED_BY', 'span_source', mention)
    set_e('token', 'belongs_to_target', 'target', tgt_belongs)
    set_e('span', 'targets', 'target', targets_e)
    set_e('span_source', 'directed_at', 'target', directed)
    if belongs:     set_e('span', 'rev_belongs', 'token', [[b, a] for a, b in belongs])
    if nested:      set_e('span_source', 'rev_NESTED_IN', 'span_source', [[b, a] for a, b in nested])
    if expresses:   set_e('span', 'rev_EXPRESSES', 'span_source', [[b, a] for a, b in expresses])
    if mention:     set_e('span_source', 'rev_MENTIONED_BY', 'token', [[b, a] for a, b in mention])
    if tgt_belongs: set_e('target', 'rev_belongs_to_target', 'token', [[b, a] for a, b in tgt_belongs])
    if targets_e:   set_e('target', 'rev_targets', 'span', [[b, a] for a, b in targets_e])
    if directed:    set_e('target', 'rev_directed_at', 'span_source', [[b, a] for a, b in directed])

    n_expr = len(expr_words)
    in_span = set(sub for w in span_w if w in w2s for sub in w2s[w])
    data['token'].aspect_mask = torch.tensor(
        [1 if (w is not None and 1 <= w <= n_expr) else 0 for w in wids], dtype=torch.long)   # prompt prefix
    data['token'].span_mask = torch.tensor(
        [1 if i in in_span else 0 for i in range(len(full_ids))], dtype=torch.long)           # inside sentence

    data.polarity = torch.tensor([sample['polarity']], dtype=torch.long)
    data.intensity = torch.tensor([sample['intensity']], dtype=torch.long)
    data.intensity5 = torch.tensor([sample['intensity5']], dtype=torch.long)       # evaluation only
    data.src_depth_orig = torch.tensor([len(sample['sources'])], dtype=torch.long)  # analysis only
    data.type_id = torch.tensor([TYPE_IDS.get(sample['type_word'], 4)], dtype=torch.long)   # input of the type embedding
    data.gold_tgt = torch.tensor([int(len(sample['targets']) > 0)], dtype=torch.long)        # analysis only
    data.span_len = torch.tensor([len(sample['span_text'].split())], dtype=torch.long)       # analysis only
    return data


class MPQADataset(Dataset):
    def __init__(self, samples):
        super().__init__()
        self.samples = samples
        print(f"[DATA] Building graphs ({MODEL_SIZE}, edges={sorted(EDGE_FAMILIES)}, "
              f"reverse={USE_REVERSE_EDGES}, sources={SOURCE_MODE})...")
        self.graphs = [build_hetero_graph(s) for s in tqdm(samples, ncols=100)]
    def __len__(self): return len(self.graphs)
    def __getitem__(self, idx): return self.graphs[idx]

## Models

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import HeteroConv, GATv2Conv
from torch_geometric.utils import to_dense_batch
from transformers import T5EncoderModel, T5ForConditionalGeneration, BitsAndBytesConfig
from peft import get_peft_model, LoraConfig, TaskType


def _bnb_config():
    return BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_compute_dtype=COMPUTE_DTYPE,
        bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4")


def _lora_config(task_type):
    return LoraConfig(task_type=task_type, inference_mode=False, r=LORA_RANK, lora_alpha=LORA_ALPHA,
                      lora_dropout=LORA_DROPOUT, target_modules=LORA_TARGETS)


def _unfreeze_lora_and_norms(module):
    """PEFT froze the base model; make sure LoRA matrices and layer norms are trainable."""
    for n, p in module.named_parameters():
        if 'lora_' in n or 'norm' in n or 'layer_norm' in n:
            p.requires_grad = True
    for p in module.parameters():                       # the optimizer and the loss scaler need fp32 weights
        if p.requires_grad and p.dtype != torch.float32:
            p.data = p.data.float()


def build_encoder():
    """FLAN-T5 encoder in 4-bit (NF4, double quantization) with LoRA adapters."""
    encoder = T5EncoderModel.from_pretrained(
        ENC['id'], quantization_config=_bnb_config(), output_hidden_states=True)
    encoder = get_peft_model(encoder, _lora_config(TaskType.FEATURE_EXTRACTION))
    encoder.print_trainable_parameters()
    return encoder, encoder.config.d_model


def build_seq2seq():
    """Full FLAN-T5 (encoder + decoder) in 4-bit with LoRA adapters, for the text-to-text baseline."""
    model = T5ForConditionalGeneration.from_pretrained(ENC['id'], quantization_config=_bnb_config())
    model = get_peft_model(model, _lora_config(TaskType.SEQ_2_SEQ_LM))
    model.print_trainable_parameters()
    _unfreeze_lora_and_norms(model)
    return model


def encode_batch(encoder, data):
    bv = data['token'].batch
    ids, _ = to_dense_batch(data['token'].input_ids, bv, fill_value=tokenizer.pad_token_id)
    mask, _ = to_dense_batch(data['token'].attention_mask, bv)
    out = encoder(input_ids=ids, attention_mask=mask, output_hidden_states=True)
    return out, mask, bv


class HeteroGATv2Layer(nn.Module):
    def __init__(self, in_dim, out_dim, edge_types, heads=2, dropout=0.1):
        super().__init__()
        cd = {}
        for s, r, d in edge_types:
            cd[(s, r, d)] = (GATv2Conv(in_dim, out_dim // heads, heads=heads, dropout=dropout, add_self_loops=True)
                             if s == d else
                             GATv2Conv((in_dim, in_dim), out_dim // heads, heads=heads, dropout=dropout,
                                       add_self_loops=False))
        self.conv = HeteroConv(cd, aggr='sum')
        self.ln = nn.LayerNorm(out_dim); self.dp = nn.Dropout(dropout)
        self.act = nn.GELU()
        self.res = nn.Linear(in_dim, out_dim) if in_dim != out_dim else nn.Identity()

    def forward(self, x, e):
        o = self.conv(x, e)
        out = {}
        for nt, feat in x.items():
            # Node types that receive no message (possible when relation families are ablated)
            # keep their residual features instead of disappearing from the graph.
            out[nt] = (self.dp(self.act(self.ln(o[nt]))) + self.res(feat)) if nt in o else self.res(feat)
        return out


class AdvancedReadout(nn.Module):
    """Span-, source-, and target-conditioned attention pooling over the token nodes."""
    def __init__(self, d, dropout=0.1):
        super().__init__()
        self.d = d
        self.span_q = nn.Linear(d, d); self.src_q = nn.Linear(d, d); self.tgt_q = nn.Linear(d, d)
        self.fusion = nn.Sequential(nn.Linear(d * 3, d), nn.LayerNorm(d), nn.GELU(), nn.Dropout(dropout))
        self.no_tgt = nn.Parameter(torch.randn(d) * 0.02)

    def forward(self, x, bv, nsrc, hastgt):
        out, off = [], 0
        sc = self.d ** 0.5
        for g, b in enumerate(torch.unique(bv)):
            gt = x['token'][bv == b]
            inner = off + nsrc[g] - 1; off += nsrc[g]
            qs = [self.span_q(x['span'][g:g + 1]), self.src_q(x['span_source'][inner:inner + 1]),
                  self.tgt_q(x['target'][g:g + 1]) if hastgt[g] else self.no_tgt.unsqueeze(0)]
            pools = []
            for q in qs:
                w = F.softmax(torch.matmul(gt, q.T).squeeze(-1) / sc, 0).unsqueeze(-1)
                pools.append((gt * w).sum(0))
            out.append(self.fusion(torch.cat(pools, -1)))
        return torch.stack(out)


class GatedFusion(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.g = nn.Sequential(nn.Linear(d * 2, d), nn.Sigmoid())
        self.p = nn.Linear(d * 2, d); self.ln = nn.LayerNorm(d)

    def forward(self, hg, hc):
        c = torch.cat([hg, hc], -1); g = self.g(c)
        return self.ln(g * self.p(c) + (1 - g) * hc)


def mlp_head(d, out_dim, dropout):
    """Two-layer MLP head: Linear -> LayerNorm -> GELU -> Dropout -> Linear."""
    return nn.Sequential(nn.Linear(d, d), nn.LayerNorm(d), nn.GELU(),
                         nn.Dropout(dropout), nn.Linear(d, out_dim))


class HS_GAT(nn.Module):
    def __init__(self, hidden_dim=256, dropout=0.1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.encoder, enc_dim = build_encoder()
        print(f"[MODEL] FLAN-T5-{MODEL_SIZE} loaded | enc_dim={enc_dim}")

        self.layer_weights = nn.Parameter(torch.ones(4) / 4)
        self.token_proj = nn.Sequential(nn.Linear(enc_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU())
        n_ctx = 3 if USE_SPAN_SKIP else 1       # sentence mean (+ aspect-token mean + in-sentence span mean)
        self.ctx_proj = nn.Sequential(nn.Linear(enc_dim * n_ctx, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU())

        self.span_query = nn.Parameter(torch.randn(1, hidden_dim) * 0.02)
        self.source_query = nn.Parameter(torch.randn(1, hidden_dim) * 0.02)
        self.target_query = nn.Parameter(torch.randn(1, hidden_dim) * 0.02)
        self.src_depth_emb = None
        if SOURCE_DEPTH_EMB:
            self.src_depth_emb = nn.Embedding(MAX_SRC_DEPTH, hidden_dim)
            nn.init.normal_(self.src_depth_emb.weight, std=0.02)

        edge_types = active_edge_types()
        self.gnn = nn.ModuleList([HeteroGATv2Layer(hidden_dim, hidden_dim, edge_types,
                                                   heads=GNN_HEADS, dropout=dropout)
                                  for _ in range(GNN_LAYERS)])
        self.final_ln = nn.LayerNorm(hidden_dim)
        self.readout = AdvancedReadout(hidden_dim, dropout)          # polarity branch (shared if not task-specific)
        self.fusion = GatedFusion(hidden_dim)
        if TASK_SPECIFIC_BRANCHES:
            self.readout_int = AdvancedReadout(hidden_dim, dropout)  # intensity branch
            self.fusion_int = GatedFusion(hidden_dim)
        else:
            self.readout_int, self.fusion_int = self.readout, self.fusion

        # Polarity: flat head + hierarchical gate (polar vs. neutral) and direction (pos vs. neg).
        self.pol_head = mlp_head(hidden_dim, NUM_POL_CLASSES, dropout)
        self.pol_gate = nn.Linear(hidden_dim, 2)
        self.pol_dir = nn.Linear(hidden_dim, 2)

        # Intensity: flat head + CORAL-style ordinal head (Cao et al., 2020).
        # One shared score s(x) and two thresholds b1 <= b2 with b2 = b1 + softplus(g),
        # so that P(u > medium) <= P(u > low) holds for every input.
        self.type_emb = None
        if USE_GOLD_TYPE and USE_TYPE_EMB:
            self.type_emb = nn.Embedding(NUM_TYPE_CLASSES, TYPE_EMB_DIM)
            self.type_to_pol = nn.Linear(TYPE_EMB_DIM, hidden_dim)
            self.type_to_int = nn.Linear(TYPE_EMB_DIM, hidden_dim)
        self.int_head = mlp_head(hidden_dim, NUM_INT_CLASSES, dropout)
        self.int_ord_score = mlp_head(hidden_dim, 1, dropout)
        self.int_ord_b1 = nn.Parameter(torch.zeros(1))
        self.int_ord_gap = nn.Parameter(torch.zeros(1))

        _unfreeze_lora_and_norms(self.encoder)

    def forward(self, data):
        out, mask, bv = encode_batch(self.encoder, data)

        hs = [h.float() for h in out.hidden_states[-4:]]
        if NORMALIZE_LAYER_MIX:
            hs = [F.layer_norm(h, h.shape[-1:]) for h in hs]               # same scale for every layer
        last_4 = torch.stack(hs, 0)                                        # (4, B, L, D)
        w = F.softmax(self.layer_weights, 0).view(4, 1, 1, 1)
        embs = (last_4 * w).sum(0)                                         # (B, L, D)
        x_token = self.token_proj(embs[mask.bool()])

        def masked_mean(msk):
            msk = msk.unsqueeze(-1).float()
            return (embs * msk).sum(1) / msk.sum(1).clamp(min=1)
        ctx = [masked_mean(mask)]                                          # T5 has no [CLS]: mean-pool
        if USE_SPAN_SKIP:
            for key in ('aspect_mask', 'span_mask'):
                dense, _ = to_dense_batch(getattr(data['token'], key), bv)
                ctx.append(masked_mean(dense))
        h_ctx = self.ctx_proj(torch.cat(ctx, -1))

        ng = bv.max().item() + 1
        nsrc = [(data['span_source'].batch == g).sum().item() for g in range(ng)] \
            if hasattr(data['span_source'], 'batch') else [1] * ng
        hastgt = data['target'].has_real_target.tolist() \
            if hasattr(data['target'], 'has_real_target') else [True] * ng

        src_feat = self.source_query.expand(sum(nsrc), -1)
        if self.src_depth_emb is not None:                                 # depth 0 = innermost source
            depth = torch.tensor([min(n - 1 - k, MAX_SRC_DEPTH - 1) for n in nsrc for k in range(n)],
                                 device=src_feat.device, dtype=torch.long)
            src_feat = src_feat + self.src_depth_emb(depth)

        x = {'token': x_token, 'span': self.span_query.expand(ng, -1),
             'span_source': src_feat, 'target': self.target_query.expand(ng, -1)}
        eid = {k: store.edge_index for k, store in data.edge_items()}
        for layer in self.gnn: x = layer(x, eid)
        x = {nt: self.final_ln(v) for nt, v in x.items()}

        h_pol = self.fusion(self.readout(x, bv, nsrc, hastgt), h_ctx)
        h_int = (self.fusion_int(self.readout_int(x, bv, nsrc, hastgt), h_ctx)
                 if self.readout_int is not self.readout else h_pol)
        if self.type_emb is not None:                                       # gold type conditions both tasks
            t_emb = self.type_emb(data.type_id)
            h_pol = h_pol + self.type_to_pol(t_emb)
            h_int = h_int + self.type_to_int(t_emb)

        score = self.int_ord_score(h_int).squeeze(-1)                      # (B,)
        b1 = self.int_ord_b1
        b2 = self.int_ord_b1 + F.softplus(self.int_ord_gap)                # b2 >= b1 always
        ord_logits = torch.stack([score - b1, score - b2], dim=1)          # [logit P(u>low), logit P(u>medium)]

        return {'pol': self.pol_head(h_pol), 'gate': self.pol_gate(h_pol),
                'dir': self.pol_dir(h_pol), 'int': self.int_head(h_int), 'ord': ord_logits}


class EncoderSoftmax(nn.Module):
    """Baseline without GNN: QLoRA encoder, mean pooling over the prompt, one softmax head per task."""
    def __init__(self, dropout=0.1):
        super().__init__()
        self.encoder, enc_dim = build_encoder()
        self.dropout = nn.Dropout(dropout)
        self.pol_head = nn.Linear(enc_dim, NUM_POL_CLASSES)
        self.int_head = nn.Linear(enc_dim, NUM_INT_CLASSES)
        _unfreeze_lora_and_norms(self.encoder)

    def forward(self, data):
        out, mask, _ = encode_batch(self.encoder, data)
        m = mask.unsqueeze(-1).float()
        h = (out.last_hidden_state.float() * m).sum(1) / m.sum(1).clamp(min=1)
        h = self.dropout(h)
        return {'pol': self.pol_head(h), 'int': self.int_head(h)}


def combine_pol_probs(p, g, d, alpha=None):
    """p_final = (1 - alpha) softmax(flat) + alpha * p_hier."""
    alpha = POL_COMBINE_ALPHA if alpha is None else alpha
    if alpha <= 0 or not USE_HIER_POL: return F.softmax(p, dim=1)
    pm, pg, pd = F.softmax(p, 1), F.softmax(g, 1), F.softmax(d, 1)
    ph = torch.stack([pg[:, 0] * pd[:, 0], pg[:, 0] * pd[:, 1], pg[:, 1]], 1)
    return (1 - alpha) * pm + alpha * ph


def ordinal_probs(ord_logits):
    """Class probabilities induced by the two cumulative thresholds."""
    p_gt1 = torch.sigmoid(ord_logits[:, 0])                                # P(u > low)
    p_gt2 = torch.sigmoid(ord_logits[:, 1])                                # P(u > medium)
    return torch.stack([1 - p_gt1, (p_gt1 - p_gt2).clamp(min=0), p_gt2], dim=1)


def combine_int_probs(i_log, ord_logits, alpha=None):
    """p_int = (1 - alpha_u) softmax(flat) + alpha_u * p_ord."""
    alpha = INT_COMBINE_ALPHA if alpha is None else alpha
    if alpha <= 0 or not USE_ORDINAL_INT: return F.softmax(i_log, dim=1)
    return (1 - alpha) * F.softmax(i_log, dim=1) + alpha * ordinal_probs(ord_logits)

## Losses

In [ ]:
def compute_class_weights(labels, num_classes, boost_idx=None, boost=1.0):
    c = torch.bincount(labels, minlength=num_classes).float()
    w = 1.0 / (c + 1e-6); w = w / w.sum() * num_classes
    if boost_idx is not None: w[boost_idx] *= boost
    return w.to(DEVICE)


def smooth_ce(logits, targets, cw, smoothing):
    """Class-weighted, label-smoothed cross-entropy (same form for polarity and intensity)."""
    if smoothing > 0:
        n = logits.size(1)
        s = torch.zeros_like(logits).fill_(smoothing / (n - 1))
        s.scatter_(1, targets.unsqueeze(1), 1.0 - smoothing)
        ce = -(s * F.log_softmax(logits, 1)).sum(1)
        if cw is not None: ce = ce * cw[targets]
        return ce.mean()
    return F.cross_entropy(logits, targets, weight=cw)


def emd_loss(logits, targets, num_classes=NUM_INT_CLASSES, soft=None):
    """L1 distance between predicted and target CDFs (earth mover's distance, ordinal)."""
    cdf = torch.cumsum(F.softmax(logits, 1), 1)[:, :-1]                    # (B, K-1)
    if soft is not None:
        tgt_cdf = torch.cumsum(soft, 1)[:, :-1]                            # CDF of the soft target
    else:
        levels = torch.arange(num_classes - 1, device=logits.device).unsqueeze(0)
        tgt_cdf = (targets.unsqueeze(1) <= levels).float()                 # 1[u <= k]
    return (cdf - tgt_cdf).abs().sum(1).mean()


# 5-way category -> distribution over (low, medium, high). Boundary categories are split half/half.
_SOFT_5TO3 = [[1., 0., 0.], [.5, .5, 0.], [0., 1., 0.], [0., .5, .5], [0., 0., 1.]]


def soft_boundary_targets(i5):
    return torch.tensor(_SOFT_5TO3, device=i5.device)[i5]


def smooth_ce_soft(logits, soft, hard, cw, smoothing):
    """Class-weighted, label-smoothed cross-entropy against a soft target distribution."""
    q = (1.0 - smoothing) * soft + smoothing / soft.size(1)
    ce = -(q * F.log_softmax(logits, 1)).sum(1)
    if cw is not None: ce = ce * cw[hard]
    return ce.mean()


def compute_loss_plain(outs, p_tgt, i_tgt, cw, i5_tgt=None, soft=None):
    """Baseline loss: the two primary terms only (no hierarchical, ordinal, or EMD terms)."""
    int_term = (smooth_ce_soft(outs['int'], soft, i_tgt, cw['intensity'], LABEL_SMOOTHING_INT) if soft is not None
                else smooth_ce(outs['int'], i_tgt, cw['intensity'], LABEL_SMOOTHING_INT))
    return (TASK_WEIGHT_POL * smooth_ce(outs['pol'], p_tgt, cw['polarity'], LABEL_SMOOTHING_POL) +
            TASK_WEIGHT_INT * int_term)


def compute_loss(outs, p_tgt, i_tgt, cw, i5_tgt=None):
    soft = soft_boundary_targets(i5_tgt) if (INT_SOFT_BOUNDARY and i5_tgt is not None) else None
    total = compute_loss_plain(outs, p_tgt, i_tgt, cw, soft=soft)
    if USE_HIER_POL:
        gate_tgt = (p_tgt == NEUTRAL_IDX).long()
        total = total + HIER_GATE_WEIGHT * F.cross_entropy(outs['gate'], gate_tgt)
        pm = gate_tgt == 0
        if pm.sum() > 0:
            total = total + HIER_DIR_WEIGHT * F.cross_entropy(outs['dir'][pm], (p_tgt[pm] == NEG_IDX).long())
    if USE_ORDINAL_INT:
        t1 = (soft[:, 1] + soft[:, 2]) if soft is not None else (i_tgt > LOW_IDX).float()      # P(u > low)
        t2 = soft[:, 2] if soft is not None else (i_tgt > MEDIUM_IDX).float()                   # P(u > medium)
        total = total + ORD_T1_WEIGHT * F.binary_cross_entropy_with_logits(outs['ord'][:, 0], t1)
        total = total + ORD_T2_WEIGHT * F.binary_cross_entropy_with_logits(outs['ord'][:, 1], t2)
    if USE_EMD_INT:
        total = total + EMD_WEIGHT * emd_loss(outs['int'], i_tgt, soft=soft)
    return total

## Eval

In [ ]:
# Evaluation. One forward pass per split is cached, and every metric (standard and calibrated
# polarity, neutral-bias search, both intensity metrics, per-source-count breakdown) is computed
# from that cache. All model types share this path.

from sklearn.metrics import f1_score, classification_report

PAPER_INTENSITY_CLASSES = ['low', 'medium', 'high', 'low-medium', 'medium-high']


def paper_intensity_weighted_f1(true5_idx, pred3_idx):
    """
    Boundary-tolerant, support-weighted intensity F1 (after Ahmadnia et al., 2024), adapted to a
    3-way model output.
    true5_idx: gold 5-way index (0=low, 1=low-medium, 2=medium, 3=medium-high, 4=high).
    pred3_idx: model 3-way prediction (0=low, 1=medium, 2=high).
    Returns (support-weighted F1, per-class breakdown).
    """
    whole, with_penalty, half_false = 1.0, 0.0, 1.0
    trues = {c: 0.0 for c in PAPER_INTENSITY_CLASSES}
    cnt = {c: 0.0 for c in PAPER_INTENSITY_CLASSES}
    falses = {c: 0.0 for c in PAPER_INTENSITY_CLASSES}

    for y5, p3 in zip(true5_idx, pred3_idx):
        y = INTENSITY_5CAT_MULTIHOT[int(y5)]
        t = (1, 0, 0) if p3 == 0 else (0, 1, 0) if p3 == 1 else (0, 0, 1)

        # false positives (feed precision)
        if t == (1, 0, 0):
            if y == (0, 0, 1): falses['low'] += 1
            if y == (0, 1, 0): falses['low'] += half_false
            if y == (0, 1, 1): falses['low'] += half_false
        elif t == (0, 1, 0):
            if y == (0, 0, 1): falses['medium'] += half_false
            if y == (1, 0, 0): falses['medium'] += half_false
        elif t == (0, 0, 1):
            if y == (1, 0, 0): falses['high'] += 1
            if y == (1, 1, 0): falses['high'] += 1
            if y == (0, 1, 1): falses['high'] += half_false
            if y == (0, 1, 0): falses['high'] += half_false

        # true positives and support (feed recall)
        if y == (1, 0, 0):
            cnt['low'] += 1
            if t == (1, 0, 0): trues['low'] += whole
        elif y == (0, 1, 0):
            cnt['medium'] += 1
            trues['medium'] += whole if t == (0, 1, 0) else with_penalty
        elif y == (0, 0, 1):
            cnt['high'] += 1
            if t == (0, 0, 1): trues['high'] += whole
        elif y == (1, 1, 0):
            cnt['low-medium'] += 1
            if t[0] == 1 or t[1] == 1: trues['low-medium'] += whole
        elif y == (0, 1, 1):
            cnt['medium-high'] += 1
            if t[1] == 1 or t[2] == 1: trues['medium-high'] += whole

    per_class, weighted_f1, weights = {}, 0.0, 0.0
    for c in PAPER_INTENSITY_CLASSES:
        precision = trues[c] / (trues[c] + falses[c]) if (trues[c] + falses[c]) > 0 else 1.0
        recall = trues[c] / cnt[c] if cnt[c] > 0 else 1.0
        f1 = statistics.harmonic_mean([precision, recall])
        per_class[c] = {'precision': precision, 'recall': recall, 'f1': f1, 'support': cnt[c]}
        weighted_f1 += f1 * cnt[c]; weights += cnt[c]
    return (weighted_f1 / weights if weights > 0 else 0.0), per_class


CACHE_KEYS = ('pol_logp', 'int_probs', 'pol', 'int', 'int5', 'nsrc')


@torch.no_grad()
def collect_predictions(model, loader):
    """Single forward pass over a split (graph-based models)."""
    model.eval()
    store = {k: [] for k in CACHE_KEYS}
    for batch in loader:
        batch = batch.to(DEVICE)
        o = model(batch)
        pol_p = combine_pol_probs(o['pol'], o['gate'], o['dir']) if 'gate' in o else F.softmax(o['pol'], 1)
        int_p = combine_int_probs(o['int'], o['ord']) if 'ord' in o else F.softmax(o['int'], 1)
        store['pol_logp'].append(torch.log(pol_p + 1e-9).cpu().numpy())
        store['int_probs'].append(int_p.cpu().numpy())
        store['pol'].append(batch.polarity.cpu().numpy())
        store['int'].append(batch.intensity.cpu().numpy())
        store['int5'].append(batch.intensity5.cpu().numpy())
        store['nsrc'].append(batch.src_depth_orig.cpu().numpy())
        for key, attr in (('type', 'type_id'), ('gold_tgt', 'gold_tgt'), ('span_len', 'span_len')):
            val = getattr(batch, attr, None)
            if val is not None: store.setdefault(key, []).append(val.cpu().numpy())
    return {k: np.concatenate(v, axis=0) for k, v in store.items() if len(v)}


def compute_metrics(P, bias=0.0, int_bias=None, return_breakdown=False):
    """Polarity at a given neutral bias, and intensity (optionally with class offsets) under both metrics."""
    logp = P['pol_logp'].copy(); logp[:, NEUTRAL_IDX] += bias
    pol_pred = logp.argmax(1)
    int_logp = np.log(P['int_probs'] + 1e-9)
    if int_bias is not None: int_logp = int_logp + np.asarray(int_bias)
    int_pred = int_logp.argmax(1)
    pc = f1_score(P['pol'], pol_pred, average=None, labels=[0, 1, 2], zero_division=0)
    int_paper, breakdown = paper_intensity_weighted_f1(P['int5'], int_pred)
    metrics = {'pol_weighted': f1_score(P['pol'], pol_pred, average='weighted', zero_division=0),
               'pol_macro': f1_score(P['pol'], pol_pred, average='macro', zero_division=0),
               'pol_neutral': pc[NEUTRAL_IDX], 'pol_pos': pc[POS_IDX], 'pol_neg': pc[NEG_IDX],
               'int_weighted': f1_score(P['int'], int_pred, average='weighted', zero_division=0),
               'int_paper_weighted_f1': int_paper}
    metrics = {k: float(v) for k, v in metrics.items()}
    if return_breakdown:
        return metrics, pol_pred, int_pred, breakdown
    return metrics, pol_pred


def calibrate_neutral_bias(P_val):
    """
    Post-hoc neutral-class threshold moving (Zhou and Liu, 2006; Buda et al., 2018).
    Grid search over CALIB_GRID on the VALIDATION cache only, maximizing macro polarity F1;
    the test split is never used. The model is not modified.
    """
    if not CALIBRATE_NEUTRAL:
        return 0.0
    logp, true = P_val['pol_logp'], P_val['pol']
    uncal = f1_score(true, logp.argmax(1), average='macro', zero_division=0)
    best_bias, best_score = 0.0, uncal
    for b in CALIB_GRID:
        shifted = logp.copy(); shifted[:, NEUTRAL_IDX] += b
        score = f1_score(true, shifted.argmax(1), average='macro', zero_division=0)
        if score > best_score:
            best_score, best_bias = score, float(b)
    print(f"  [CALIB] neutral bias = {best_bias:+.2f} | val macro F1: "
          f"standard(b=0)={uncal:.4f} -> calibrated={best_score:.4f}")
    return best_bias


def error_analysis(P, title="validation"):
    """Where does polarity fail? Confusion matrix, then errors by gold type, gold target and span length.
    The gold type is an input of the model (see USE_GOLD_TYPE); here it is only used to group the errors."""
    from sklearn.metrics import confusion_matrix
    true, pred = P['pol'], P['pol_logp'].argmax(1)
    names = ['pos', 'neg', 'neu']
    cm = confusion_matrix(true, pred, labels=[0, 1, 2])
    print(f"\n  [ERROR ANALYSIS | polarity | {title} | standard decision rule]")
    print("    confusion matrix (rows: gold, columns: predicted; row %)")
    print("           " + "".join(f"{n:>14s}" for n in names))
    for i, n in enumerate(names):
        tot = max(1, cm[i].sum())
        print(f"    {n:>5s}  " + "".join(f"{cm[i, j]:>7d} ({100 * cm[i, j] / tot:4.1f}%)" for j in range(3)))
    total_err = max(1, int((true != pred).sum()))

    def group_report(label, groups):
        print(f"    by {label}: n | accuracy | share of all errors | gold label mix (pos/neg/neu %)")
        for gname, mask in groups:
            n = int(mask.sum())
            if n < 20: continue
            acc = float((true[mask] == pred[mask]).mean())
            err_share = 100.0 * float((true[mask] != pred[mask]).sum()) / total_err
            mix = [100.0 * float((true[mask] == c).mean()) for c in range(3)]
            print(f"      {gname:<14s} {n:>5d} | {acc:.3f} | {err_share:5.1f}% | "
                  f"{mix[0]:4.1f} / {mix[1]:4.1f} / {mix[2]:4.1f}")
    if 'type' in P:
        tn = ['agreement', 'arguing', 'subjectivity', 'intention', 'sentiment']
        group_report("gold opinion type", [(tn[t], P['type'] == t) for t in range(5)])
    if 'gold_tgt' in P:
        group_report("annotated target", [('has target', P['gold_tgt'] == 1), ('no target', P['gold_tgt'] == 0)])
    if 'span_len' in P:
        L = P['span_len']
        group_report("expression length", [('1 word', L == 1), ('2-3 words', (L >= 2) & (L <= 3)), ('4+ words', L >= 4)])


def _int_selection_score(true3, true5, pred):
    return (SEL_W_INT_STD * f1_score(true3, pred, average='weighted', zero_division=0)
            + SEL_W_INT_PAPER * paper_intensity_weighted_f1(true5, pred)[0])


def calibrate_intensity_bias(P_val):
    """
    Validation-tuned log-offsets (0, b_medium, b_high) added to the intensity log-probabilities.
    Grid search on the VALIDATION cache only, maximizing the intensity selection score
    (0.5 weighted 3-way F1 + 0.5 boundary-tolerant F1); the test split is never used.
    """
    if not CALIBRATE_INTENSITY:
        return None
    logp = np.log(P_val['int_probs'] + 1e-9)
    base = _int_selection_score(P_val['int'], P_val['int5'], logp.argmax(1))
    best, best_off = base, np.zeros(3)
    for bm in INT_CALIB_GRID:
        for bh in INT_CALIB_GRID:
            off = np.array([0.0, bm, bh])
            sc = _int_selection_score(P_val['int'], P_val['int5'], (logp + off).argmax(1))
            if sc > best + 1e-6:
                best, best_off = sc, off
    print(f"  [CALIB] intensity offsets (medium, high) = ({best_off[1]:+.1f}, {best_off[2]:+.1f}) | "
          f"val selection score: standard={base:.4f} -> calibrated={best:.4f}")
    return best_off


def depth_breakdown(P_pol, P_int, bias):
    """Test metrics by number of annotated nested sources (1, 2, 3 or more)."""
    buckets = [('1 source', lambda n: n == 1), ('2 sources', lambda n: n == 2),
               ('3+ sources', lambda n: n >= 3)]
    rows = {}
    for name, cond in buckets:
        mp, mi = cond(P_pol['nsrc']), cond(P_int['nsrc'])
        if mp.sum() == 0: continue
        sub_p = {k: v[mp] for k, v in P_pol.items()}
        sub_i = {k: v[mi] for k, v in P_int.items()}
        mp_, _ = compute_metrics(sub_p, bias)
        mi_, _ = compute_metrics(sub_i)
        rows[name] = {'n': int(mp.sum()), 'pol_weighted': mp_['pol_weighted'], 'pol_macro': mp_['pol_macro'],
                      'int_weighted': mi_['int_weighted'], 'int_paper_weighted_f1': mi_['int_paper_weighted_f1']}
    print("  [By number of nested sources | polarity calibrated, intensity standard]")
    for name, r in rows.items():
        print(f"    {name:<11s} n={r['n']:<5d} Pol(W) {r['pol_weighted']:.4f}  Pol(M) {r['pol_macro']:.4f}  "
              f"Int(W) {r['int_weighted']:.4f}  Int(Paper W) {r['int_paper_weighted_f1']:.4f}")
    return rows

## Runners

In [ ]:
# Runners wrap what differs between model families (data pipeline, model, optimizer,
# training epoch, prediction); the fold loop is shared.

from torch_geometric.loader import DataLoader
from torch.utils.data import DataLoader as TorchDataLoader


class EMA:
    """Exponential moving average of the trainable parameters (fp32 shadow copy)."""
    def __init__(self, model, decay):
        self.decay, self.n = decay, 0
        self.shadow = {n: p.detach().clone() for n, p in model.named_parameters() if p.requires_grad}

    @torch.no_grad()
    def update(self, model):
        self.n += 1
        d = min(self.decay, (1 + self.n) / (10 + self.n))              # short warm-up of the average
        for n, p in model.named_parameters():
            if n in self.shadow:
                self.shadow[n].mul_(d).add_(p.detach(), alpha=1 - d)

    @contextlib.contextmanager
    def average(self, model):
        """Temporarily load the averaged weights (for validation, checkpointing, testing)."""
        backup = {}
        with torch.no_grad():
            for n, p in model.named_parameters():
                if n in self.shadow:
                    backup[n] = p.detach().clone(); p.copy_(self.shadow[n])
        try:
            yield
        finally:
            with torch.no_grad():
                for n, p in model.named_parameters():
                    if n in backup: p.copy_(backup[n])


def make_scaler():
    return torch.amp.GradScaler('cuda', enabled=USE_GRAD_SCALER)


LAST_GRAD_NORM = float('nan')       # mean pre-clipping gradient norm of the last epoch (diagnostic)


def run_train_epoch(model, loader, loss_fn, optimizer, scheduler, desc, scaler=None, ema=None):
    """One epoch with gradient accumulation; loss_fn(batch) returns the (unscaled) batch loss."""
    global LAST_GRAD_NORM
    scaler = scaler if scaler is not None else make_scaler()
    params = [p for p in model.parameters() if p.requires_grad]
    model.train()
    tot, n, gnorms = 0.0, 0, []
    optimizer.zero_grad()
    pbar = tqdm(loader, desc=desc, ncols=110)
    for step, batch in enumerate(pbar):
        loss = loss_fn(batch) / GRAD_ACCUM
        scaler.scale(loss).backward()
        if (step + 1) % GRAD_ACCUM == 0 or (step + 1) == len(loader):
            scaler.unscale_(optimizer)
            gn = torch.nn.utils.clip_grad_norm_(params, max_norm=GRAD_CLIP)
            if torch.isfinite(gn): gnorms.append(float(gn))
            scaler.step(optimizer); scaler.update()
            scheduler.step(); optimizer.zero_grad()
            if ema is not None: ema.update(model)
        tot += loss.item() * GRAD_ACCUM; n += 1
        pbar.set_postfix({'Loss': f"{tot/n:.4f}"})
    LAST_GRAD_NORM = float(np.mean(gnorms)) if gnorms else float('nan')
    return tot / n


def build_optimizer(model):
    """Two parameter groups: encoder-side (LoRA + norms) and GNN/heads."""
    enc_params, head_params = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad: continue
        (enc_params if n.startswith('encoder.') else head_params).append(p)
    groups = [{'params': enc_params, 'lr': ENC_BASE_LR, 'weight_decay': WEIGHT_DECAY_ENC}]
    if head_params:
        groups.append({'params': head_params, 'lr': GNN_LR, 'weight_decay': WEIGHT_DECAY_HEAD})
    return torch.optim.AdamW(groups)


class GraphRunner:
    """HS-GAT and the encoder-only softmax baseline (both consume the graph dataset)."""
    supports_calibration = True

    def __init__(self, train_s, val_s, test_s):
        train_ds, val_ds, test_ds = MPQADataset(train_s), MPQADataset(val_s), MPQADataset(test_s)
        g = torch.Generator(); g.manual_seed(SEED)
        self.loaders = {'train': DataLoader(train_ds, BATCH_SIZE, shuffle=True, generator=g),
                        'val': DataLoader(val_ds, BATCH_SIZE, shuffle=False),
                        'test': DataLoader(test_ds, BATCH_SIZE, shuffle=False)}
        self.cw = {'polarity': compute_class_weights(torch.tensor([s['polarity'] for s in train_s]),
                                                     NUM_POL_CLASSES, NEUTRAL_IDX, NEUTRAL_BOOST),
                   'intensity': compute_class_weights(torch.tensor([s['intensity'] for s in train_s]),
                                                      NUM_INT_CLASSES)}
        self.loss_fn_impl = compute_loss if MODEL_TYPE == 'hsgat' else compute_loss_plain

    def n_batches(self): return len(self.loaders['train'])

    def build_model(self):
        if MODEL_TYPE == 'hsgat':
            return HS_GAT(HIDDEN_DIM, DROPOUT_RATE).to(DEVICE)
        return EncoderSoftmax(DROPOUT_RATE).to(DEVICE)

    def build_optimizer(self, model): return build_optimizer(model)

    def train_epoch(self, model, optimizer, scheduler, epoch, fold):
        def loss_fn(batch):
            batch = batch.to(DEVICE)
            return self.loss_fn_impl(model(batch), batch.polarity, batch.intensity, self.cw, batch.intensity5)
        return run_train_epoch(model, self.loaders['train'], loss_fn, optimizer, scheduler,
                               f"Fold {fold} | Ep {epoch:02d}/{EPOCHS}", getattr(self, 'scaler', None),
                               getattr(self, 'ema', None))

    def predict(self, model, split):
        return collect_predictions(model, self.loaders[split])


# Text-to-text baseline
_POL_RE = re.compile(r"polarity\s*:\s*(positive|negative|neutral)", re.I)
_INT_RE = re.compile(r"intensity\s*:\s*(low|medium|high)", re.I)


def t2t_source(sample):
    typ = f" | type= {sample['type_word']}" if USE_GOLD_TYPE else ""
    return f"{T2T_INSTRUCTION} aspect= {sample['span_text']} | sentence= {sample['text']}{typ}"


def t2t_target(sample):
    return f"polarity: {POL_WORDS[sample['polarity']]} ; intensity: {INT_WORDS[sample['intensity']]}"


def parse_t2t(text, fallback_pol, fallback_int):
    """Returns (polarity idx, intensity idx, parsed_ok). Unparsable fields fall back to the majority class."""
    mp, mi = _POL_RE.search(text), _INT_RE.search(text)
    pol = POL_WORDS.index(mp.group(1).lower()) if mp else fallback_pol
    inten = INT_WORDS.index(mi.group(1).lower()) if mi else fallback_int
    return pol, inten, bool(mp and mi)


class T2TCollator:
    def __init__(self, with_labels): self.with_labels = with_labels

    def __call__(self, samples):
        enc = tokenizer([t2t_source(s) for s in samples], padding=True, truncation=True,
                        max_length=MAX_SEQ_LEN, return_tensors='pt')
        out = {'input_ids': enc['input_ids'], 'attention_mask': enc['attention_mask']}
        if self.with_labels:
            lab = tokenizer([t2t_target(s) for s in samples], padding=True, truncation=True,
                            max_length=T2T_MAX_NEW_TOKENS * 2, return_tensors='pt')['input_ids']
            lab[lab == tokenizer.pad_token_id] = -100
            out['labels'] = lab
        return out


class T2TRunner:
    """Text-to-text baseline: FLAN-T5 + QLoRA generates 'polarity: .. ; intensity: ..'."""
    supports_calibration = False

    def __init__(self, train_s, val_s, test_s):
        self.samples = {'train': train_s, 'val': val_s, 'test': test_s}
        g = torch.Generator(); g.manual_seed(SEED)
        self.loaders = {
            'train': TorchDataLoader(train_s, BATCH_SIZE, shuffle=True, generator=g, collate_fn=T2TCollator(True)),
            'val': TorchDataLoader(val_s, BATCH_SIZE * 2, shuffle=False, collate_fn=T2TCollator(False)),
            'test': TorchDataLoader(test_s, BATCH_SIZE * 2, shuffle=False, collate_fn=T2TCollator(False))}
        self.fb_pol = int(np.bincount([s['polarity'] for s in train_s], minlength=3).argmax())
        self.fb_int = int(np.bincount([s['intensity'] for s in train_s], minlength=3).argmax())
        self.parse_fail_rate = 0.0

    def n_batches(self): return len(self.loaders['train'])

    def build_model(self): return build_seq2seq()

    def build_optimizer(self, model):
        params = [p for p in model.parameters() if p.requires_grad]
        return torch.optim.AdamW(params, lr=ENC_BASE_LR, weight_decay=WEIGHT_DECAY_ENC)

    def train_epoch(self, model, optimizer, scheduler, epoch, fold):
        def loss_fn(batch):
            b = {k: v.to(DEVICE) for k, v in batch.items()}
            return model(**b).loss
        return run_train_epoch(model, self.loaders['train'], loss_fn, optimizer, scheduler,
                               f"Fold {fold} | Ep {epoch:02d}/{EPOCHS} [t2t]", getattr(self, 'scaler', None),
                               getattr(self, 'ema', None))

    @torch.no_grad()
    def predict(self, model, split):
        model.eval()
        texts = []
        for batch in self.loaders[split]:
            gen = model.generate(input_ids=batch['input_ids'].to(DEVICE),
                                 attention_mask=batch['attention_mask'].to(DEVICE),
                                 max_new_tokens=T2T_MAX_NEW_TOKENS, num_beams=1, do_sample=False)
            texts += tokenizer.batch_decode(gen, skip_special_tokens=True)
        samples = self.samples[split]
        parsed = [parse_t2t(t, self.fb_pol, self.fb_int) for t in texts]
        self.parse_fail_rate = 1.0 - float(np.mean([p[2] for p in parsed]))
        pol_pred = np.array([p[0] for p in parsed]); int_pred = np.array([p[1] for p in parsed])
        # Hard predictions are expressed in the shared cache format as (almost) one-hot scores.
        return {'pol_logp': np.log(np.eye(3)[pol_pred] + 1e-9),
                'int_probs': np.eye(3)[int_pred].astype(float),
                'pol': np.array([s['polarity'] for s in samples]),
                'int': np.array([s['intensity'] for s in samples]),
                'int5': np.array([s['intensity5'] for s in samples]),
                'nsrc': np.array([len(s['sources']) for s in samples])}

## Checkpoints and Fold Loop

In [ ]:
def save_checkpoint(model, path):
    """QLoRA-safe: keep LoRA, heads, and norms; skip the 4-bit base weights and bnb metadata."""
    sd = {k: v for k, v in model.state_dict().items()
          if not any(t in k for t in QUANT_META)
          and not (('base_layer.weight' in k) and ('lora' not in k))}
    torch.save(sd, path)
    print(f"  [CKPT] saved {len(sd)} tensors")


def load_checkpoint(model, path):
    sd = torch.load(path, weights_only=True, map_location=DEVICE)
    missing, unexpected = model.load_state_dict(sd, strict=False)
    print(f"  [CKPT] loaded | missing (frozen base, expected): {len(missing)} | unexpected: {len(unexpected)}")
    return model


METRIC_KEYS = ['pol_weighted', 'pol_macro', 'pol_neutral', 'pol_pos', 'pol_neg',
               'int_weighted', 'int_paper_weighted_f1']


def run_fold(fold_idx, id_to_sample, folds):
    set_seed()
    print(f"\n{'#'*80}\n#  FOLD {fold_idx+1}/{NUM_FOLDS} | {MODEL_NAME} | seed {SEED}\n{'#'*80}")
    sp = folds[fold_idx]
    train_s = [id_to_sample[i] for i in sp['train_ids'] if i in id_to_sample]
    val_s = [id_to_sample[i] for i in sp['val_ids'] if i in id_to_sample]
    test_s = [id_to_sample[i] for i in sp['test_ids'] if i in id_to_sample]
    if not (train_s and val_s and test_s):
        return None
    print(f"  Train: {len(train_s)} | Val: {len(val_s)} | Test: {len(test_s)}")

    runner = (T2TRunner if MODEL_TYPE == 'text2text' else GraphRunner)(train_s, val_s, test_s)
    runner.scaler = make_scaler()
    model = runner.build_model()
    if fold_idx == FOLDS_TO_RUN[0]:
        print(f"[MODEL] Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
    optimizer = runner.build_optimizer(model)
    steps_per_epoch = (runner.n_batches() + GRAD_ACCUM - 1) // GRAD_ACCUM
    ema_decay = 0.5 ** (1.0 / max(1.0, EMA_HALF_LIFE_EPOCHS * steps_per_epoch))
    ema = EMA(model, ema_decay) if USE_EMA else None
    runner.ema = ema
    if ema is not None: print(f"  EMA decay per step = {ema_decay:.5f} (half-life {EMA_HALF_LIFE_EPOCHS} epoch)")

    steps = ((runner.n_batches() + GRAD_ACCUM - 1) // GRAD_ACCUM) * EPOCHS
    ws = int(steps * WARMUP_RATIO)
    def lr_lambda(s):
        if s < ws: return s / max(1, ws)
        pr = (s - ws) / max(1, steps - ws)
        return COSINE_ETA_MIN_RATIO + (1 - COSINE_ETA_MIN_RATIO) * 0.5 * (1 + np.cos(np.pi * pr))
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    # Task-specific checkpoints: polarity and intensity need not peak at the same epoch,
    # so each keeps its own best-on-validation checkpoint (selected at neutral bias 0).
    best_pol, best_pol_ep, best_int, best_int_ep = -1.0, 0, -1.0, 0
    since_improve = 0
    tag = f"{MODEL_NAME}_s{SEED}_fold_{fold_idx+1}"
    bp_pol, bp_int = f"{CHECKPOINT_DIR}/best_{tag}_pol.pth", f"{CHECKPOINT_DIR}/best_{tag}_int.pth"

    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()
        improved = False
        runner.train_epoch(model, optimizer, scheduler, epoch, fold_idx + 1)
        with (ema.average(model) if ema is not None else contextlib.nullcontext()):
            m, _ = compute_metrics(runner.predict(model, 'val'))
            sel_pol = SEL_W_POL_WEIGHTED * m['pol_weighted'] + SEL_W_POL_MACRO * m['pol_macro']
            sel_int = SEL_W_INT_STD * m['int_weighted'] + SEL_W_INT_PAPER * m['int_paper_weighted_f1']
            extra = f" | parse-fail {runner.parse_fail_rate:.3f}" if MODEL_TYPE == 'text2text' else ""
            print(f"  Ep {epoch}/{EPOCHS} | Pol(W): {m['pol_weighted']:.4f} | Pol(M): {m['pol_macro']:.4f} | "
                  f"NEUT: {m['pol_neutral']:.4f} | Int(W): {m['int_weighted']:.4f} | "
                  f"Int(Paper W): {m['int_paper_weighted_f1']:.4f} | sel_pol {sel_pol:.4f} | "
                  f"sel_int {sel_int:.4f} | gnorm {LAST_GRAD_NORM:.2f} | {time.time()-t0:.0f}s{extra}")
            if sel_pol > best_pol:
                best_pol, best_pol_ep, improved = sel_pol, epoch, True
                save_checkpoint(model, bp_pol); print("       best POLARITY checkpoint")
            if sel_int > best_int:
                best_int, best_int_ep, improved = sel_int, epoch, True
                save_checkpoint(model, bp_int); print("       best INTENSITY checkpoint")
        since_improve = 0 if improved else since_improve + 1
        if EARLY_STOP_PATIENCE and since_improve >= EARLY_STOP_PATIENCE:
            print(f"  Early stop after epoch {epoch}: no validation improvement for {since_improve} epochs.")
            break

    print(f"\n  Best checkpoints (fold {fold_idx+1}): POLARITY @ep{best_pol_ep} "
          f"(sel={best_pol:.4f}) | INTENSITY @ep{best_int_ep} (sel={best_int:.4f})")

    # Polarity: polarity-best checkpoint, one forward pass per split
    load_checkpoint(model, bp_pol)
    P_val_pol = runner.predict(model, 'val')
    bias = calibrate_neutral_bias(P_val_pol) if runner.supports_calibration else 0.0
    if runner.supports_calibration:
        error_analysis(P_val_pol, 'validation')       # diagnostics on validation, never on test
    P_test_pol = runner.predict(model, 'test')
    m_std, pred_std = compute_metrics(P_test_pol, bias=0.0)
    m_cal, pred_cal = compute_metrics(P_test_pol, bias=bias)

    # Intensity: intensity-best checkpoint
    load_checkpoint(model, bp_int)
    int_off = calibrate_intensity_bias(runner.predict(model, 'val')) if runner.supports_calibration else None
    P_test_int = runner.predict(model, 'test')
    m_int, _, int_pred, breakdown = compute_metrics(P_test_int, return_breakdown=True)
    m_int_cal = compute_metrics(P_test_int, int_bias=int_off)[0] if int_off is not None else m_int

    f = fold_idx + 1
    print(f"\n  TEST F{f} POLARITY STANDARD   (b= 0.00): Pol(W) {m_std['pol_weighted']:.4f} | "
          f"Pol(M) {m_std['pol_macro']:.4f} | NEUT {m_std['pol_neutral']:.4f} | "
          f"pos {m_std['pol_pos']:.4f} | neg {m_std['pol_neg']:.4f}")
    if runner.supports_calibration:
        print(f"  TEST F{f} POLARITY CALIBRATED (b={bias:+.2f}): Pol(W) {m_cal['pol_weighted']:.4f} | "
              f"Pol(M) {m_cal['pol_macro']:.4f} | NEUT {m_cal['pol_neutral']:.4f} | "
              f"pos {m_cal['pol_pos']:.4f} | neg {m_cal['pol_neg']:.4f}")
    print(f"  TEST F{f} INTENSITY STANDARD:   Int(W) {m_int['int_weighted']:.4f} | "
          f"Int(Paper W) {m_int['int_paper_weighted_f1']:.4f}")
    if int_off is not None:
        print(f"  TEST F{f} INTENSITY CALIBRATED: Int(W) {m_int_cal['int_weighted']:.4f} | "
              f"Int(Paper W) {m_int_cal['int_paper_weighted_f1']:.4f}")
    names = ['positive', 'negative', 'neutral']
    print("  [Polarity report | standard]")
    print(classification_report(P_test_pol['pol'], pred_std, labels=[0, 1, 2], target_names=names, zero_division=0))
    print("  [Intensity report | 3-way, standard]")
    print(classification_report(P_test_int['int'], int_pred, labels=[0, 1, 2],
                                target_names=['low', 'medium', 'high'], zero_division=0))
    print("  [Intensity | boundary-tolerant metric, per category]")
    for c, s in breakdown.items():
        print(f"    {c:<12s} P={s['precision']:.4f}  R={s['recall']:.4f}  "
              f"F1={s['f1']:.4f}  support={int(s['support'])}")
    depth_rows = depth_breakdown(P_test_pol, P_test_int, bias)

    # Each field comes from its own best checkpoint; calibration affects polarity only.
    std = {**{k: m_std[k] for k in METRIC_KEYS if k.startswith('pol')},
           'int_weighted': m_int['int_weighted'], 'int_paper_weighted_f1': m_int['int_paper_weighted_f1']}
    cal = {**{k: m_cal[k] for k in METRIC_KEYS if k.startswith('pol')},
           'int_weighted': m_int_cal['int_weighted'], 'int_paper_weighted_f1': m_int_cal['int_paper_weighted_f1']}

    del model, optimizer, scheduler, runner
    gc.collect(); torch.cuda.empty_cache()
    return {'fold': fold_idx + 1, 'seed': SEED, 'experiment': EXPERIMENT, 'bias': bias,
            'int_bias': (None if int_off is None else [float(x) for x in int_off]),
            'std': std, 'cal': cal, 'by_nsrc': depth_rows}

## Results

In [ ]:
def summarize(runs, title):
    print(f"\n{'='*80}\n  {title}\n{'='*80}")
    for k in METRIC_KEYS:
        v = [r['std'][k] for r in runs]
        vc = [r['cal'][k] for r in runs]
        print(f"  {k:24s}: std {np.mean(v):.4f} +/- {np.std(v):.4f} | cal {np.mean(vc):.4f} +/- {np.std(vc):.4f}")


def collect_results_table(results_dir=None, reference='full'):
    """Aggregate all result JSON files into one ablation table, with paired Wilcoxon tests vs. `reference`."""
    results_dir = results_dir or RESULTS_DIR
    by_exp = {}
    for path in sorted(glob.glob(os.path.join(results_dir, "*.json"))):
        with open(path) as f:
            for r in json.load(f):
                by_exp.setdefault(r['experiment'], []).append(r)
    if not by_exp:
        print("[RESULTS] no result files found"); return
    cols = [('Pol W', 'std', 'pol_weighted'), ('Pol M', 'std', 'pol_macro'), ('Pol M cal', 'cal', 'pol_macro'),
            ('Int W', 'std', 'int_weighted'), ('Int Paper', 'std', 'int_paper_weighted_f1')]
    order = [e for e in EXPERIMENTS if e in by_exp] + [e for e in by_exp if e not in EXPERIMENTS]
    print(f"\n{'experiment':<24s}" + "".join(f"{c[0]:>17s}" for c in cols) + "   runs")
    for e in order:
        runs = by_exp[e]
        cells = "".join(f"{np.mean([r[s][k] for r in runs]):>10.4f}+-{np.std([r[s][k] for r in runs]):.3f}" for _, s, k in cols)
        print(f"{e:<24s}{cells}   {len(runs)}")
    try:
        from scipy.stats import wilcoxon
        if reference not in by_exp: return
        ref = {(r['seed'], r['fold']): r for r in by_exp[reference]}
        print(f"\nPaired Wilcoxon signed-rank p-values vs. '{reference}' (over matched seed x fold runs)")
        for e in order:
            if e == reference: continue
            pairs = [(r, ref[(r['seed'], r['fold'])]) for r in by_exp[e] if (r['seed'], r['fold']) in ref]
            if len(pairs) < 5:
                print(f"  {e:<24s} too few matched runs ({len(pairs)})"); continue
            line = f"  {e:<24s}"
            for name, s, k in cols:
                a, b = [p[0][s][k] for p in pairs], [p[1][s][k] for p in pairs]
                try: pv = wilcoxon(a, b).pvalue
                except ValueError: pv = float('nan')
                line += f"  {name}: p={pv:.3f}"
            print(line)
    except ImportError:
        print("[RESULTS] scipy not installed; skipping significance tests")

## Main

In [ ]:
def main():
    queue = [e for e in os.environ.get('HSGAT_EXPERIMENTS', '').split(',') if e] or EXPERIMENT_QUEUE
    id_to_sample = load_all_data()
    folds = load_fold_splits()
    for name in queue:
        for seed in SEED_LIST:
            apply_experiment(name)
            globals()['SEED'] = seed
            init_tools()
            set_seed()
            runs = []
            for fold_idx in FOLDS_TO_RUN:
                res = run_fold(fold_idx, id_to_sample, folds)
                if res is not None: runs.append(res)
            if runs:
                summarize(runs, f"{len(runs)}-FOLD AVERAGES | {MODEL_NAME} | seed {seed}")
                out = os.path.join(RESULTS_DIR, f"{name}_seed{seed}.json")
                with open(out, 'w') as f: json.dump(runs, f, indent=1)
                print(f"  Results saved to {out}")
    print("  Polarity metrics come from the polarity-best checkpoint, "
          "intensity metrics from the intensity-best checkpoint.")
    collect_results_table()



if __name__ == "__main__":
    main()